In [3]:
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
greedy_affine = "/home/tuguldur/Development/Research/Dev/ECNDP/ECNDP/python/results/csv/Result_ECNDP_ER_all_aff_only.csv"
others = "/home/tuguldur/Development/Research/Dev/ECNDP/ECNDP/python/results/csv/Result_ECNDP_random_ER_all_affinity.csv"


In [4]:
df_1 = pd.read_csv(greedy_affine)
df_2 = pd.read_csv(others)
df_1

,case,algo,terminals,non_terminals,K,obj,time
0,2,Greedy affinity - no ls,9,91,9,36.0,0.02034
1,2,Greedy affinity - ls,9,91,9,21.0,0.32348
2,2,Greedy affinity - no ls,9,91,18,36.0,0.02088
3,2,Greedy affinity - ls,9,91,18,21.0,0.56133
4,2,Greedy affinity - no ls,9,91,27,36.0,0.02039
5,2,Greedy affinity - ls,9,91,27,21.0,0.72303
6,2,Greedy affinity - no ls,9,91,36,36.0,0.02226
7,2,Greedy affinity - ls,9,91,36,21.0,0.81405
8,2,Greedy affinity - no ls,9,91,45,36.0,0.02094
9,2,Greedy affinity - ls,9,91,45,21.0,0.83899


In [5]:
df_affinity = df_1[df_1["algo"].str.contains("Greedy affinity")]

# from df_2 keep only ES and MIS
df_other = df_2[
  df_2["algo"].str.contains("Greedy ES") |
  df_2["algo"].str.contains("Greedy MIS")
]


In [6]:
# combine
df_all = pd.concat([df_affinity, df_other], ignore_index=True)

# optional: sort
df_all = df_all.sort_values(["terminals", "algo", "K"])

In [8]:
df_all["K"] = pd.to_numeric(df_all["K"], errors="coerce")

# keep only integer K values
df_all = df_all[df_all["K"] % 1 == 0]

# convert to int for clean plotting
df_all["K"] = df_all["K"].astype(int)

In [12]:
def clean_name(algo_name):
  if "ES" in algo_name:
    base = "Greedy"
  elif "MIS" in algo_name:
    base = "MIS"
  else:
    base = "Affinity"

  if "no ls" in algo_name:
    return base + " (no LS)"
  else:
    return base + " (LS)"

In [14]:
import os
import matplotlib.pyplot as plt

# create folder for plots
output_folder = "/home/tuguldur/Development/Research/Dev/ECNDP/ECNDP/python/results/plots"
os.makedirs(output_folder, exist_ok=True)

terminal_values = [9, 19, 30, 40, 50]

for terminal in terminal_values:
  df_terminal = df_all[df_all["terminals"] == terminal]

  plt.figure()

  for algo_name in df_terminal["algo"].unique():
    df_algo = df_terminal[df_terminal["algo"] == algo_name]
    df_algo = df_algo.sort_values("K")

    plt.plot(
      df_algo["K"],
      df_algo["obj"],
      marker="o",
      label=clean_name(algo_name)
    )

  k_values = sorted(df_terminal["K"].unique())
  plt.xticks(k_values)

  plt.title(f"Objective vs K (terminals = {terminal})")
  plt.xlabel("K")
  plt.ylabel("Objective (obj)")
  plt.legend()
  plt.grid()

  # save file
  filename = f"{output_folder}/terminals_{terminal}.png"
  plt.savefig(filename, dpi=300, bbox_inches="tight")

  plt.close()  # important to avoid overlap